## 06 - Cross-Source Coverage
Cel: analiza jak źródła danych współgrają ze sobą — gdzie się pokrywają, gdzie są dziury, co dashboard realnie pokaże.
Tabele: gold.sentiment_vs_returns, gold.sentiment_lead_lag, gold.av_sentiment_sector_daily, gold.macro_impact_on_tech

In [0]:
%sql
SELECT 
  o.symbol,
  COUNT(DISTINCT o.date) AS trading_days,
  COUNT(DISTINCT s.date) AS sentiment_days,
  ROUND(COUNT(DISTINCT s.date) * 100.0 / COUNT(DISTINCT o.date), 2) AS coverage_pct
FROM silver.ohlcv_indicators o
LEFT JOIN gold.av_sentiment_aggregated s
ON o.symbol = s.symbol AND o.date = s.date
WHERE o.symbol IN (SELECT DISTINCT symbol FROM gold.av_sentiment_aggregated)
GROUP BY o.symbol
ORDER BY coverage_pct ASC

In [0]:
%sql
SELECT 
  o.symbol,
  COUNT(DISTINCT s.date) AS sentiment_days,
  ROUND(COUNT(DISTINCT s.date) * 100.0 / COUNT(DISTINCT o.date), 2) AS coverage_pct
FROM silver.ohlcv_indicators o
INNER JOIN gold.av_sentiment_aggregated s
ON o.symbol = s.symbol AND o.date = s.date
GROUP BY o.symbol
ORDER BY sentiment_days DESC
LIMIT 10

In [0]:
%sql
SELECT 
  year_month,
  monthly_return,
  CASE WHEN cpiaucsl IS NULL THEN 'MISSING' ELSE 'OK' END AS cpi_status,
  CASE WHEN fedfunds IS NULL THEN 'MISSING' ELSE 'OK' END AS fedfunds_status,
  CASE WHEN dgs10 IS NULL THEN 'MISSING' ELSE 'OK' END AS dgs10_status,
  CASE WHEN unrate IS NULL THEN 'MISSING' ELSE 'OK' END AS unrate_status
FROM gold.macro_impact_on_tech
ORDER BY year_month

In [0]:
%sql
SELECT 
  d.industry,
  d.total_days AS sentiment_sector_days,
  COUNT(DISTINCT r.symbol) AS symbols_with_returns
FROM (
  SELECT industry, COUNT(*) AS total_days
  FROM gold.av_sentiment_sector_daily
  GROUP BY industry
) d
LEFT JOIN gold.ohlcv_with_dimension o
ON d.industry = o.industry
LEFT JOIN gold.sentiment_vs_returns r
ON o.symbol = r.symbol
GROUP BY d.industry, d.total_days
ORDER BY sentiment_sector_days DESC

### Wnioski
1. Pokrycie sentymentem dni handlowych bardzo rzadkie - najlepszy NVDA 11.32%, mediana ~6-7%. Dashboard sentiment vs returns operuje na rzadkich danych, wnioski to wskazówki, nie twarde zależności.
2. Top 10 symboli z najgęstszym pokryciem: NVDA (30 dni), GFS (29), ON (28), MSFT (28), MRVL (24), ADP (23), AMD (21), AMAT (21), ROP (21), ASML (20). Te dają najbardziej wiarygodne wyniki.
3. 11/13 miesięcy macro impact ma kompletne dane. Październik 2025 brak CPI i UNRATE (źródło FRED). Marzec 2026 brak CPI, FEDFUNDS, UNRATE (bieżący miesiąc).
4. Semiconductors (103 dni, 14 symboli) i software-application (114 dni, 12 symboli) - wiarygodne na dashboardzie per industry. Communication equipment i consumer electronics po 1 symbolu - agregacja sektorowa bezwartościowa.
5. Dashboard powinien: fokusować się na top 10 symboli z najlepszym pokryciem, oznaczać niekompletne miesiące makro, filtrować industry z <5 symbolami.
